# Part B — Customer Segments and Lapse Prediction

This notebook completes all eight Part B tasks using the reusable source modules in `part_b/`. Run Part A first so that `data/clean/orders.csv`, `customers.csv` and `products.csv` exist, then run these cells from top to bottom. All model calls are local scikit-learn operations; no API key is needed. Tables and printed results are used throughout; no charts or images are generated.

The RFM snapshot is **2025-10-01 00:00:00**, and lapse-model features stop strictly before **2025-09-01 00:00:00**. Recency and tenure are whole days; monetary values are in **INR**. Reproducibility uses `random_state=42`, K-means `n_init=10`, and the prescribed train/test split and parameter grids.

In [1]:
from pathlib import Path
import os
import sys
import json

# Works when the notebook starts from the repository root or from part_b/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "part_b" / "rfm.py").is_file() and (p / "data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the project root or part_b folder.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from part_b.rfm import compute_rfm_table
from part_b.clustering import evaluate_kmeans_clusters, build_segments
from part_b.lapse_modeling import (
    NUMERIC_COLUMNS, CATEGORICAL_COLUMNS, FEATURE_COLUMNS, LR_GRID, RF_GRID,
    build_cohort_and_features, split_lapse_data, make_preprocessor,
    balance_training_only, run_lapse_experiment, RETENTION_LIMITATION,
)
from part_b.run_part_b import CLUSTER_CHOICE, write_part_b_report

for name in ["orders.csv", "customers.csv", "products.csv"]:
    assert (ROOT / "data" / "clean" / name).is_file(), "Run Part A before Part B."
print("Project root:", ROOT)
print("Required cleaned tables exist. All monetary values are INR.")

Project root: /home/sanjeevtg/Documents/MASAIASSIGNMENTS/Capstone _Project
Required cleaned tables exist. All monetary values are INR.


## Task 1 — Build the RFM table

For every customer with at least one completed order before the snapshot, compute `recency = (snapshot − last_completed_order_ts).days`, `frequency = number of distinct completed order IDs`, and `monetary = sum of completed order_amount`. Cancelled orders do not count toward purchasing behaviour. Orders at or after the snapshot cannot enter an as-of table.

In [2]:
rfm = compute_rfm_table(snapshot_date="2025-10-01 00:00:00")
print(rfm.head(10).to_string(index=False))
rfm_summary = pd.DataFrame([{
    "customers": len(rfm), "recency_median": rfm.recency.median(),
    "recency_min": rfm.recency.min(), "recency_max": rfm.recency.max(),
    "frequency_mean": rfm.frequency.mean(), "frequency_max": rfm.frequency.max(),
    "monetary_mean_inr": rfm.monetary.mean(), "monetary_max_inr": rfm.monetary.max(),
}])
print("\nRFM checkpoint:\n", rfm_summary.to_string(index=False))
assert len(rfm) == 1175
assert (rfm.recency.median(), rfm.recency.min(), rfm.recency.max()) == (19, 0, 180)
assert np.isclose(rfm.frequency.mean(), 9.46, atol=0.01)
assert np.isclose(rfm.monetary.mean(), 2552.06, atol=0.02)

customer_id  recency  frequency  monetary
      C0001       14          4    512.85
      C0002       28          7   4644.90
      C0003        7         22   3644.85
      C0004        0          5    369.00
      C0005      147          2    324.00
      C0006       34          1     90.00
      C0007        3         18   2600.00
      C0008        3         15   3096.50
      C0009       19         11   8424.80
      C0010       66          4    450.00

RFM checkpoint:
  customers  recency_median  recency_min  recency_max  frequency_mean  frequency_max  monetary_mean_inr  monetary_max_inr
      1175            19.0            0          180        9.455319             33        2552.058553          14766.55


## Task 2 — Standardise RFM and compare K=2…8

RFM variables use different units, so `StandardScaler` first gives each column zero mean and unit variance. Each candidate then uses `KMeans(n_clusters=K, n_init=10, random_state=42)`. The table is the elbow evidence; the final choice must be in {3, 4, 5}.

In [3]:
kmeans_evaluation, rfm_scaler, scaled_rfm = evaluate_kmeans_clusters(rfm)
print(kmeans_evaluation.to_string(index=False))
print("\nInertia reductions when adding a cluster:")
reductions = kmeans_evaluation.copy()
reductions["inertia_reduction"] = -reductions.inertia.diff()
print(reductions.to_string(index=False))
chosen_k = 4
assert chosen_k in {3, 4, 5}
print("\n", CLUSTER_CHOICE)
assert np.all(np.diff(kmeans_evaluation.inertia) < 0)

 k  inertia  silhouette
 2   1820.8       0.465
 3   1268.2       0.445
 4    743.1       0.526
 5    617.4       0.480
 6    503.2       0.482
 7    408.5       0.416
 8    366.5       0.412

Inertia reductions when adding a cluster:
 k  inertia  silhouette  inertia_reduction
 2   1820.8       0.465                NaN
 3   1268.2       0.445              552.6
 4    743.1       0.526              525.1
 5    617.4       0.480              125.7
 6    503.2       0.482              114.2
 7    408.5       0.416               94.7
 8    366.5       0.412               42.0

 Choose K=4: the large inertia reduction from K=3 to K=4 is followed by smaller improvements, and K=4 has the strongest silhouette among the actionable choices K=3,4,5. Its four profiles support distinct occasional-buyer, premium, frequent-shopper and win-back campaigns. K-means inertia is the within-cluster squared-distance objective; adding centres increases flexibility, so the best achievable inertia cannot increa

**Why K=4 and why inertia alone is insufficient.** The change from K=3 to K=4 is a substantial elbow improvement, followed by smaller gains; K=4 also has the best silhouette among the business choices. Four distinct profiles support campaigns the retention team can actually operate. Inertia measures within-cluster squared distance: more centres provide more flexibility, so the best achievable value cannot increase, and the measured K=8 inertia is lower than K=2. That fact alone cannot select K because increasing complexity can make segments fragmented and difficult to act on.

## Task 3 — Name, profile, mark Premium and project the segments

Cluster numbers are arbitrary labels with **no ordering**. Premium is exactly the cluster with the highest mean monetary value. Each profile receives one business name and a recommended campaign action. PCA uses the **scaled RFM matrix** and reports the variance retained by two components; this projection is descriptive and does not enter the lapse features.

In [4]:
segments, segment_profile, pca_ratios = build_segments(rfm, k=chosen_k)
print(segment_profile.to_string(index=False))
print("\nPCA component variance ratios:", pca_ratios)
print("Total explained variance:", float(pca_ratios.sum()))
premium_id = int(segment_profile.loc[segment_profile.mean_monetary.idxmax(), "segment_id"])
assert segment_profile.is_premium.sum() == 1
assert segments.is_premium.equals(segments.segment_id.eq(premium_id))
assert np.allclose(pca_ratios, [0.634, 0.218], atol=0.02)
expected_segment_columns = ["customer_id", "recency", "frequency", "monetary", "segment_id", "segment_name", "is_premium"]
assert pd.read_csv("outputs/customer_segments.csv").columns.tolist() == expected_segment_columns
print("\nWrote outputs/customer_segments.csv; Premium segment ID:", premium_id)

 segment_id  size  mean_recency  mean_frequency  mean_monetary                segment_name                                                         recommended_action  is_premium
          0   436     28.862385        3.772936    1057.618234 Occasional / Regular Buyers     Suggest relevant replenishment bundles to increase purchase frequency.       False
          1   138     24.115942        6.768116    7129.695652          High-Value Premium         Offer a priority support desk and personalised high-value bundles.        True
          2   419      8.766110       19.403341    3410.020525    Active Frequent Shoppers Reward repeat purchases with loyalty benefits and replenishment reminders.       False
          3   182    109.043956        2.203297     685.992582           Dormant / At-Risk       Send a time-limited win-back offer and investigate service problems.       False

PCA component variance ratios: [0.63427875 0.21803762]
Total explained variance: 0.8523163667502167

Wrote ou

## Task 4 — Define lapse and engineer strictly pre-cutoff features

The cohort contains every customer with a completed order before **2025-09-01**. The positive label `lapsed=1` means no completed order in the interval **2025-09-01 inclusive through 2025-10-01 exclusive**. September orders define labels only; later months cannot change this September target.

Features are recency, frequency, monetary, average completed-order discount percentage, cancelled orders divided by all orders with a final status (`completed` or `cancelled`), average completed-order delivery minutes, most frequent completed-order product category, city, age, gender and tenure days. Category ties use the first sorted mode for reproducibility. Clean customer attributes are joined by customer ID. Whole-window segment IDs, names and Premium flags are excluded from features.

In [5]:
features = build_cohort_and_features(cutoff_date="2025-09-01 00:00:00")
print(features[["customer_id", *FEATURE_COLUMNS, "lapsed"]].head(5).to_string(index=False))
print("\nCohort rows:", len(features))
print("Lapsed:", int(features.lapsed.sum()), "percentage:", round(features.lapsed.mean() * 100, 2))
print("Feature time audit:", features.attrs)
assert len(features) == 1156 and int(features.lapsed.sum()) == 443
assert pd.Timestamp(features.attrs["maximum_feature_order_ts"]) < pd.Timestamp("2025-09-01")
assert not {"segment_id", "segment_name", "is_premium"}.intersection(FEATURE_COLUMNS)
assert not features[FEATURE_COLUMNS].isna().any().any()
print("No missing values in any feature column:", not features[FEATURE_COLUMNS].isna().any().any())
X_train, X_test, y_train, y_test = split_lapse_data(features)
print("Train/test rows:", len(X_train), "/", len(X_test))
print("Training label counts:", y_train.value_counts().sort_index().to_dict())
print("Test label counts:", y_test.value_counts().sort_index().to_dict())
assert (len(X_train), len(X_test)) == (924, 232)
assert set(X_train.index).isdisjoint(X_test.index)

customer_id  recency  frequency  monetary  avg_discount_pct  cancellation_rate  avg_delivery_minutes  age  tenure_days favourite_category      city gender  lapsed
      C0001       33          2    378.00         10.000000           0.000000             28.000000 35.0          597            Grocery   Chennai   male       0
      C0002       22          6   3946.90          6.666667           0.000000             21.916667 31.0          562        Electronics     Delhi   male       0
      C0003        1         20   3264.05         12.750000           0.000000             24.200000 18.0          195            Grocery Bengaluru female       0
      C0004       42          3    272.25         11.666667           0.400000             35.666667 24.0          437             Snacks Bengaluru female       0
      C0005      117          2    324.00         25.000000           0.333333             26.000000 25.0          335              Dairy Bengaluru female       1

Cohort rows: 1156
Lap

## Task 5 — Preprocess without leakage

`ColumnTransformer` scales numeric columns and applies `OneHotEncoder(drop="first")` to the nominal columns. Every supervised model places this transformer inside its pipeline, so GridSearchCV fits preprocessing anew inside each training fold. The test rows are transformed only by fitted training transformers. The following preview illustrates the same training-only transformations without fitting a classifier.

In [6]:
preview_preprocessor = make_preprocessor()
encoded_train = preview_preprocessor.fit_transform(X_train)
encoded_test = preview_preprocessor.transform(X_test)
learned_frequency_mean = preview_preprocessor.named_transformers_["num"].mean_[NUMERIC_COLUMNS.index("frequency")]
full_cohort_frequency_mean = features.frequency.mean()
print("Training scaler frequency mean:", learned_frequency_mean)
print("Full-cohort frequency mean:", full_cohort_frequency_mean)
print("They differ because the scaler uses training customers only, while the full-cohort mean also includes held-out customers.")
print("Encoded train/test shapes:", encoded_train.shape, encoded_test.shape)
print("Learned features:", preview_preprocessor.get_feature_names_out().tolist())
encoder = preview_preprocessor.named_transformers_["cat"]
expected_nominal_columns = sum(len(categories) - 1 for categories in encoder.categories_)
assert encoded_train.shape[1] == len(NUMERIC_COLUMNS) + expected_nominal_columns
assert encoded_test.shape[1] == encoded_train.shape[1]
assert not X_train.isna().any().any() and not X_test.isna().any().any()

Training scaler frequency mean: 8.114718614718615
Full-cohort frequency mean: 8.028546712802768
They differ because the scaler uses training customers only, while the full-cohort mean also includes held-out customers.
Encoded train/test shapes: (924, 19) (232, 19)
Learned features: ['num__recency', 'num__frequency', 'num__monetary', 'num__avg_discount_pct', 'num__cancellation_rate', 'num__avg_delivery_minutes', 'num__age', 'num__tenure_days', 'cat__favourite_category_Electronics', 'cat__favourite_category_Grocery', 'cat__favourite_category_Personal Care', 'cat__favourite_category_Snacks', 'cat__city_Chennai', 'cat__city_Delhi', 'cat__city_Hyderabad', 'cat__city_Mumbai', 'cat__city_Pune', 'cat__gender_male', 'cat__gender_unknown']


## Task 6 — Baseline, tuning, untouched-test evaluation and model choice

The reusable experiment first tunes **both models on training rows only**, then evaluates the held-out test rows. Logistic regression uses `max_iter=1000`, five-fold F1 scoring and C ∈ {0.01, 0.1, 1, 10, 100}: 25 CV fits. Random forest uses 100 trees, seed 42, max_depth ∈ {3, 5, 8} and min_samples_leaf ∈ {5, 10, 20}: 45 CV fits. Best estimators are refitted on all training rows.

The selected base model is chosen by best training cross-validated F1 before test scoring. The majority baseline predicts `not lapsed` everywhere; precision, recall and F1 below always refer to the positive **lapsed** class, and metric calls use `zero_division=0`. Confusion-matrix rows are actual [not lapsed, lapsed], columns predicted [not lapsed, lapsed]. The experiment also computes the balance and persistence audits inspected in Tasks 7 and 8.

In [7]:
print("Logistic regression grid:", LR_GRID)
print("Random forest grid:", RF_GRID)
results = run_lapse_experiment(features)
print("\nMajority baseline:", results["baseline"])
assert results["baseline"]["f1"] == 0
for model_key, label in [("lr", "Logistic regression"), ("rf", "Random forest")]:
    print("\n", label, "best training-CV results:", results[model_key + "_tuning"])
    model_metrics = results[model_key + "_test"]
    print("Lapsed-class precision/recall/F1:", {name: model_metrics[name] for name in ["precision", "recall", "f1"]})
    print("Confusion matrix:\n", pd.DataFrame(model_metrics["cm"], index=["actual_not_lapsed", "actual_lapsed"], columns=["predicted_not_lapsed", "predicted_lapsed"]).to_string())
print("\nModel selection and campaign costs:")
print(results["selected_model_justification"])
selected_key = "rf" if results["selected_model"] == "random forest" else "lr"
assert results[selected_key + "_test"]["f1"] >= 0.55
assert np.isclose(results["learned_freq_mean"], learned_frequency_mean)

Logistic regression grid: {'clf__C': [0.01, 0.1, 1, 10, 100]}
Random forest grid: {'clf__max_depth': [3, 5, 8], 'clf__min_samples_leaf': [5, 10, 20]}

Majority baseline: {'cm': [[143, 0], [89, 0]], 'accuracy': 0.6163793103448276, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

 Logistic regression best training-CV results: {'grid': {'clf__C': [0.01, 0.1, 1, 10, 100]}, 'best_params': {'clf__C': 100}, 'best_cv_f1': 0.650552962900789, 'cv_fits': 25}
Lapsed-class precision/recall/F1: {'precision': 0.6444444444444445, 'recall': 0.651685393258427, 'f1': 0.6480446927374302}
Confusion matrix:
                    predicted_not_lapsed  predicted_lapsed
actual_not_lapsed                   111                32
actual_lapsed                        31                58

 Random forest best training-CV results: {'grid': {'clf__max_depth': [3, 5, 8], 'clf__min_samples_leaf': [5, 10, 20]}, 'best_params': {'clf__max_depth': 8, 'clf__min_samples_leaf': 10}, 'best_cv_f1': 0.686504101416853, 'cv_fits': 45}
L

## Task 7 — Oversample the training minority only

Keep **every original training row**, and add randomly selected lapsed training duplicates with replacement (`random_state=42`) until both classes have equal counts. The selected logistic-regression C is unchanged and its fresh pipeline is refitted on balanced training data. The original test set is kept intact. Precision captures wasted targeting, while recall captures how many truly lapsing customers the retention team reaches.

In [8]:
X_balanced_check, y_balanced_check = balance_training_only(X_train, y_train)
pd.testing.assert_frame_equal(X_balanced_check.iloc[:len(X_train)], X_train)
assert set(X_balanced_check.index).isdisjoint(X_test.index)
print("Original training counts:", results["train_class_counts"])
print("Balanced training counts:", results["balanced_train_class_counts"])
print("Extra lapsed duplicates:", results["oversampled_duplicate_count"])
comparison = pd.DataFrame([
    {"training": "original", **{m: results["lr_test"][m] for m in ["precision", "recall", "f1"]}},
    {"training": "balanced", **{m: results["lr_balanced_test"][m] for m in ["precision", "recall", "f1"]}},
])
print("\nSame original test set:\n", comparison.to_string(index=False))
print("Changes:", results["balance_metric_changes"])
print(results["balance_explanation"])
assert y_balanced_check.value_counts().nunique() == 1

Original training counts: {'0': 570, '1': 354}
Balanced training counts: {'0': 570, '1': 570}
Extra lapsed duplicates: 216

Same original test set:
 training  precision   recall       f1
original   0.644444 0.651685 0.648045
balanced   0.589147 0.853933 0.697248
Changes: {'precision': -0.05529715762273901, 'recall': 0.2022471910112359, 'f1': 0.04920301368458824}
Recall moves most (+0.2022); balancing increases the incentive to identify lapsing customers, but extra targeting can lower precision.


## Task 8 — Interpret, persist and use the selected model

Random-forest importance reports relative split utility, not a causal effect or coefficient direction. Logistic coefficients are conditional associations: numeric coefficients act on standardised features, and one-hot coefficients compare with the dropped reference category. Correlated behaviour measures can change coefficient signs, so causal claims require additional evidence.

In [9]:
print("Top five random-forest features and business readings:")
print(pd.DataFrame(results["top5_features"]).to_string(index=False))
print("\nThree logistic-regression coefficient signs and business readings:")
print(pd.DataFrame(results["lr_coefficient_readings"]).to_string(index=False))

# The experiment dumped the selected fitted pipeline and directly compared
# reloaded predictions with its original in-memory test predictions.
reloaded_model = joblib.load("outputs/lapse_model.joblib")
assert results["reload_predictions_match"]
assert len(reloaded_model.predict(X_test)) == len(y_test)
print("\nReload reproduces the original test predictions exactly:", results["reload_predictions_match"])
retention = pd.read_csv("outputs/retention_list.csv")
print("Premium customers in test set:", results["premium_test_customers"])
print("\nTop predicted lapse probabilities among Premium test customers:")
print(retention.to_string(index=False))
print("Actually lapsed in this list:", results["retention_actually_lapsed_count"], "/", len(retention))
assert retention.columns.tolist() == ["customer_id", "lapse_probability", "actually_lapsed"]
assert len(retention) == min(10, results["premium_test_customers"])
assert retention.lapse_probability.is_monotonic_decreasing
assert retention.lapse_probability.between(0, 1).all()
print("\n", RETENTION_LIMITATION)

Top five random-forest features and business readings:
             feature  importance                                                        business_reading
            monetary    0.277830 Earlier spending captures customer value and purchase intensity in INR.
           frequency    0.276787       Earlier completed-order frequency distinguishes regular shoppers.
             recency    0.111219             Days since the last completed order indicate disengagement.
avg_delivery_minutes    0.061649                  Earlier delivery duration captures service experience.
    avg_discount_pct    0.053178     Earlier discount usage helps identify promotion-sensitive shoppers.

Three logistic-regression coefficient signs and business readings:
         feature  coefficient     sign                                                                                                                                                        business_reading
         recency     0.039585 positive  

**Retention-list limitation.** The Premium flag is a post-hoc whole-window RFM label that includes September; therefore the retention list is illustrative and is **not a leakage-free evaluation**. The model itself uses only pre-cutoff order features. For a prospective campaign, recompute segments strictly before the prediction cutoff and evaluate against a later, unseen outcome window.

No encoder, scaler or resampler saw test rows during fitting. No order dated 2025-09-01 onward entered a model feature, and no whole-window segment label was used as a feature. Before deployment, monitor drift and campaign conversion, validate probability calibration and business costs, and repeat this evaluation across several temporal cutoffs.

In [10]:
report = write_part_b_report(rfm, kmeans_evaluation, segment_profile, pca_ratios, results)
# Built-in JSON types only; this is also produced by python part_b/run_part_b.py.
json.dumps(report, allow_nan=False)
for output in ["customer_segments.csv", "lapse_model.joblib", "retention_list.csv", "part_b_report.json"]:
    path = ROOT / "outputs" / output
    assert path.is_file() and path.stat().st_size > 0
    print("Verified:", path)
print("Part B completed: all required exports exist, model F1 >= 0.55, and reload parity holds.")

Verified: /home/sanjeevtg/Documents/MASAIASSIGNMENTS/Capstone _Project/outputs/customer_segments.csv
Verified: /home/sanjeevtg/Documents/MASAIASSIGNMENTS/Capstone _Project/outputs/lapse_model.joblib
Verified: /home/sanjeevtg/Documents/MASAIASSIGNMENTS/Capstone _Project/outputs/retention_list.csv
Verified: /home/sanjeevtg/Documents/MASAIASSIGNMENTS/Capstone _Project/outputs/part_b_report.json
Part B completed: all required exports exist, model F1 >= 0.55, and reload parity holds.
